In [ ]:
import logging
import os
import numpy as np
import open3d as o3d
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
import cv2

from marmopose.version import __version__ as marmopose_version
from marmopose.config import Config

from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.preprocessing import StandardScaler
import umap

from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy.ndimage import gaussian_filter
from scipy.interpolate import RegularGridInterpolator

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info(f'MarmoPose version: {marmopose_version}')

from marmopose.utils.data_io import load_points_3d_h5
from marmopose.utils.data_io import load_points_bboxes_2d_h5


In [ ]:
os.chdir('..')
config_path = '../configs/default.yaml'

config = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/single',
)
os.chdir('umap')
idx_spinemid = config.animal['bodyparts'].index('spinemid')
idx_tailbase = config.animal['bodyparts'].index('tailbase')
idx_neck = config.animal['bodyparts'].index('neck')


In [ ]:
# Empty arrays where collected data will be concatenated
points_3d = np.empty(shape=(1,0,16,3))
rotated_points_3d = np.empty(shape=(0,16,3))
inputs = np.empty(shape=(0,48))
# inputs = np.empty(shape=(0,96))
rotated_points_3d_velocity = np.empty(shape=(0,16,3))

# Empty lists where collected data will be appended
videos = []
n_frames = []
frames_included = []
dirs = ['260513/Home','260513/Etho']
frames_alldirs = []
# For all sessions
# for i in range(24):
for i in dirs:
    frames_alldirs.append(np.empty(0))
    # Check if there are valid videos and frame sequences for this session
    SRC_DIR = f"/scratch/VideoTracking/Videos/{i}"
    # SRC_DIR = f"/scratch/VideoTracking/Videos/Test3.{i}"
    if not os.path.isdir(SRC_DIR):
        print(SRC_DIR)
        continue
    frames_idx_SRC_DIR = f"/srv/MarmOT/VideoTracking/Videos/{i}"
    # frames_idx_SRC_DIR = f"/srv/MarmOT/VideoTracking/Videos/Test3.{i}"
    if os.path.exists(os.path.join(frames_idx_SRC_DIR,"input/frames_experiment.txt")):
        path_f_exp = os.path.join(frames_idx_SRC_DIR,"input/frames_experiment.txt")
    elif os.path.exists(os.path.join(frames_idx_SRC_DIR,"Input/frames_experiment.txt")):
        path_f_exp = os.path.join(frames_idx_SRC_DIR,"Input/frames_experiment.txt")
    else:
        path_f_exp = None
    print(path_f_exp)
    # Load 3d points for this session
    points_3d_video = load_points_3d_h5(os.path.join(SRC_DIR,"Output/points_3d/optimized.h5"))

    # Get all the frame sequences for this session 
    if not path_f_exp is None:
        with open(path_f_exp) as fp:
            frames = [[int(frame) for frame in line.strip().split(" ")] for line in fp]
    else:
        frames = [[0,points_3d_video.shape[1]]]

    # Compute 3d positions relative to spinemid point
    points_3d_ = points_3d_video - points_3d_video[:,:,[idx_spinemid],:]

    # Orient all points so that the body is aligned to the x-axis
    vector_body = points_3d_[:,:,[idx_neck],:] - points_3d_[:,:,[idx_tailbase],:]
    magnitudes_vector_body_xy = np.sqrt(vector_body[0,:,0,0]**2 + vector_body[0,:,0, 1]**2)[:,np.newaxis,np.newaxis]
    rotation_matrices = np.array([[vector_body[0,:,0,0],vector_body[0,:,0,1]],[-vector_body[0,:,0,1],vector_body[0,:,0,0]]]).transpose((2,0,1))/magnitudes_vector_body_xy
    rotated_points_3d_video = np.copy(points_3d_)[0,:,:,:]
    rotated_points_3d_video[:,:,:2] = np.einsum('ikl,ijl -> ijk', rotation_matrices, rotated_points_3d_video[:,:,:2])

    # Compute velocity based on frames n + 1
    velocity_video = points_3d_video[0,1:,:,:] - points_3d_video[0,:-1,:,:]
    # Rotate velocity
    rotated_points_3d_velocity_video = np.copy(velocity_video)
    rotated_points_3d_velocity_video[:,:,:2] = np.einsum('ikl,ijl -> ijk', rotation_matrices[:-1,:,:], velocity_video[:,:,:2])
    # Load bboxes for this session
    bbox_video = load_points_bboxes_2d_h5(os.path.join(SRC_DIR,"Output/points_2d/original.h5"), [f"output{i+1}" for i in range(4)])[1]

    # Loop over frame sequences for this session
    for f_start, f_end in frames:
        end = min(f_end, points_3d_video.shape[1])
        # Get frames where at least 2 cameras detect a marmoset
        frames_2bboxes = f_start + np.nonzero(np.sum(np.isnan(bbox_video[:,:,f_start:end - 1,0]),axis=0) < 3)[1] # -1 because velocity can't be computed in the last frame
        # Get data for those frames
        points_3d_fseq = points_3d_video[:,frames_2bboxes,:,:]
        rotated_points_3d_fseq = rotated_points_3d_video[frames_2bboxes,:,:]
        rotated_points_3d_velocity_fseq = rotated_points_3d_velocity_video[frames_2bboxes,:,:]
        # Append data to lists and arrays
        videos.append(f"/scratch/VideoTracking/Videos/{i}")
        n_frames.append(frames_2bboxes.size) 
        frames_included.append(frames_2bboxes) 
        points_3d = np.concatenate((points_3d,points_3d_fseq),axis=1)
        rotated_points_3d = np.concatenate((rotated_points_3d,rotated_points_3d_fseq),axis=0)
        rotated_points_3d_velocity = np.concatenate((rotated_points_3d_velocity,rotated_points_3d_velocity_fseq), axis=0)
        # Reshape to 2D vector and concatenate into 2D input array
        rotated_points_3d_ = rotated_points_3d_fseq.reshape((-1,48))
        rotated_points_3d_velocity_ = rotated_points_3d_velocity_fseq.reshape(-1,48)
        inputs_video = np.concatenate((rotated_points_3d_,rotated_points_3d_velocity_), axis = 1)

        # Concatenate inputs for this frame sequence to those from other frame sequences
        # inputs = np.concatenate((inputs,inputs_video),axis=0)
        inputs = np.concatenate((inputs,rotated_points_3d_),axis=0)
        frames_alldirs[-1] = np.concatenate((frames_alldirs[-1], np.arange(f_start, end - 1)), axis = 0)
        
# Compute cumulative frame number
n_frames_cumsum = np.cumsum(n_frames)


In [ ]:
scaler = StandardScaler()
normalized_inputs = scaler.fit_transform(inputs)
pca = PCA(n_components=0.95)
inputs_pca = pca.fit_transform(normalized_inputs)
print(f"Number of principal components retained: {inputs_pca.shape[1]}")
print(pca.explained_variance_ratio_)
fig = plt.figure()
ax = fig.add_subplot(111)
ax.scatter(inputs_pca[:,0],inputs_pca[:,1])
fig.show()

from sklearn.covariance import MinCovDet
import numpy as np

def robust_pca(X, n_components):
    robust_cov = MinCovDet().fit(X)
    cov_matrix = robust_cov.covariance_
    eigenvals, eigenvecs = np.linalg.eigh(cov_matrix)
    idx = np.argsort(eigenvals)[::-1]
    if n_components < 1:
        cumulative_variance = np.cumsum(eigenvals[idx]) / np.sum(eigenvals)
        n_components = np.argmax(cumulative_variance >= n_components) + 1
    components = eigenvecs[:, idx[:n_components]]
    return X @ components, eigenvals[idx[:n_components]]/np.sum(eigenvals)

inputs_pca, eigenvals = robust_pca(normalized_inputs,0.95)
print(f"Number of principal components retained: {inputs_pca.shape[1]}")
print(eigenvals)
print(np.sum(eigenvals))
fig = plt.figure()
ax = fig.add_subplot(111)
ax.scatter(inputs_pca[:,0],inputs_pca[:,1])
fig.show()



In [ ]:
%matplotlib inline
plt.clf()
selected_idx_pca = np.random.choice(inputs_pca.shape[0],10000)
# selected_pca = inputs_pca[selected_idx_pca, :]
selected_pca = inputs_pca
for nn in range(30,40,10):
    X_umap = umap.UMAP(
        n_neighbors=nn,
        min_dist=0,
        n_components=2,
        random_state=10
    ).fit_transform(selected_pca)

    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.scatter(X_umap[:,0],X_umap[:,1])
    fig.show()


In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
print(X_umap.shape)
chosen_idx = np.random.choice(np.arange(X_umap.shape[0]),5000,replace=False)
chosen_umap = X_umap[chosen_idx, :]
for i in range(2, 12):
    for j in range(5,30,5):
        clustering = DBSCAN(eps=i, min_samples=j)
        cluster_labels = clustering.fit_predict(chosen_umap)
        if len(np.unique(cluster_labels)) == 1:
            break 
        silhouette_avg = silhouette_score(chosen_umap, cluster_labels)
        silhouette_scores.append(silhouette_avg)
        print(f"For eps = {i} and min_samples = {j}, the silhouette score is: {silhouette_avg}")
        np.random.seed(3)
        fig = plt.figure()
        ax = fig.add_subplot(111)
        ax.set_title(f'EPS = {i}, Min_samples = {j}, silhouette score = {silhouette_avg}, ncluster = {len(np.unique(cluster_labels)) + np.min(cluster_labels)}')
        c = np.random.rand(len(np.unique(cluster_labels)),3)

        ax.scatter(chosen_umap[:,0],chosen_umap[:,1], c=c[cluster_labels - np.min(cluster_labels)])
        fig.show()

n_clusters_opt = np.argmax(silhouette_scores) + 2

plt.plot(range(2, 2 + len(silhouette_scores)), silhouette_scores, marker='o')
plt.title('Silhouette Scores')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score
%matplotlib inline
silhouette_scores = []
for i in range(10, 150,20):
    for j in range(2,11,2):
        clustering = HDBSCAN(min_cluster_size=i, min_samples=j)
        cluster_labels = clustering.fit_predict(chosen_umap)
        if len(np.unique(cluster_labels)) == 1:
            break 
        silhouette_avg = silhouette_score(chosen_umap, cluster_labels)
        silhouette_scores.append(silhouette_avg)
        print(f"For eps = {i} and min_samples = {j}, the silhouette score is: {silhouette_avg}")
        np.random.seed(3)
        fig = plt.figure()
        ax = fig.add_subplot(111)
        ax.set_title(f'EPS = {i}, Min_samples = {j}, silhouette score = {silhouette_avg}, ncluster = {len(np.unique(cluster_labels)) + np.min(cluster_labels)}')
        c = np.random.rand(len(np.unique(cluster_labels)),3)

        ax.scatter(chosen_umap[:,0],chosen_umap[:,1], c=c[cluster_labels - np.min(cluster_labels)])
        fig.show()

n_clusters_opt = np.argmax(silhouette_scores) + 2

plt.plot(range(2, 2 + len(silhouette_scores)), silhouette_scores, marker='o')
plt.title('Silhouette Scores')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.show()


In [ ]:
DBscan = DBSCAN(eps=2, min_samples=20)
clusters = DBscan.fit_predict(X_umap)
n_clusters_opt = len(np.unique(clusters)) + np.min(clusters)


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
np.random.seed(3)
plt.clf()
# Create a 3D scatter plot
fig = plt.figure()
ax = fig.add_subplot(111)
c = np.random.rand(len(np.unique(clusters))  - np.min(clusters),3)
for i in np.unique(clusters):
    cluster_label = 'Noise' if i == -1 else f'Cluster {i}'
    idx_cluster_i = np.nonzero(clusters == i)[0]
    ax.scatter(X_umap[idx_cluster_i,0],X_umap[idx_cluster_i,1], c=c[i - np.min(clusters)], label = cluster_label)
    if i != -1:
        ax.text(np.mean(X_umap[idx_cluster_i,0]),np.mean(X_umap[idx_cluster_i,1]),i)
# ax.set_xlim((None,300))
# ax.legend()
fig.show()

In [ ]:
%matplotlib inline
for size in range(50,501,50):
    HDBscan = HDBSCAN(min_cluster_size=50, min_samples=size)
    clusters = HDBscan.fit_predict(X_umap)
    n_clusters_opt = len(np.unique(clusters)) + np.min(clusters)

    np.random.seed(3)
    # Create a 3D scatter plot
    fig = plt.figure()
    ax = fig.add_subplot(111)
    c = np.random.rand(len(np.unique(clusters))  - np.min(clusters),3)
    for i in np.unique(clusters):
        cluster_label = 'Noise' if i == -1 else f'Cluster {i}'
        idx_cluster_i = np.nonzero(clusters == i)[0]
        ax.scatter(X_umap[idx_cluster_i,0],X_umap[idx_cluster_i,1], c=c[i - np.min(clusters)], label = cluster_label)
        if i != -1:
            ax.text(np.mean(X_umap[idx_cluster_i,0]),np.mean(X_umap[idx_cluster_i,1]),i)
    ax.set_title(f'Min_cluster_size={size}')
    fig.show()

In [ ]:
# DBscan = DBSCAN(eps=7, min_samples=35)
# clusters = DBscan.fit_predict(X_tsne)
# n_clusters_opt = len(np.unique(clusters)) + np.min(clusters)

# HDBscan = HDBSCAN(min_cluster_size=150, min_samples=15,cluster_selection_epsilon=0.35)
HDBscan = HDBSCAN(min_cluster_size=60,min_samples=10)
clusters = HDBscan.fit_predict(chosen_umap)
n_clusters_opt = len(np.unique(clusters)) + np.min(clusters)

fig = plt.figure()
ax = fig.add_subplot(111)
ax.set_title(f'EPS = {i}, Min_samples = {j}, silhouette score = {silhouette_avg}, ncluster = {len(np.unique(cluster_labels)) + np.min(cluster_labels)}')
c = np.random.rand(len(np.unique(cluster_labels)),3)

ax.scatter(chosen_umap[:,0],chosen_umap[:,1], c=c[cluster_labels - np.min(cluster_labels)])
fig.show()



In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
np.random.seed(3)
# Create a 3D scatter plot
fig = plt.figure()
ax = fig.add_subplot(111)
c = np.random.rand(len(np.unique(clusters))  - np.min(clusters),3)
for i in np.unique(clusters):
    cluster_label = 'Noise' if i == -1 else f'Cluster {i}'
    idx_cluster_i = np.nonzero(clusters == i)[0]
    ax.scatter(X_umap[idx_cluster_i,0],X_umap[idx_cluster_i,1], c=c[i - np.min(clusters)], label = cluster_label)
    if i != -1:
        ax.text(np.mean(X_umap[idx_cluster_i,0]),np.mean(X_umap[idx_cluster_i,1]),i)
# ax.set_xlim((None,300))
# ax.legend()
fig.show()

In [ ]:
# Create a 2D histogram of the t-SNE points
hist, xedges, yedges = np.histogram2d(X_umap[:, 0], X_umap[:, 1], bins=100, range=[[X_umap[:, 0].min(), X_umap[:, 0].max()], [X_umap[:, 1].min(), X_umap[:, 1].max()]])

# Smooth the histogram with a Gaussian filter
density = gaussian_filter(hist, sigma=4,mode='constant')
# Find local maxima (peaks) in the density map to use as markers
peaks = peak_local_max(density, min_distance=2)
# Label the peaks for watershed
markers = np.zeros_like(density, dtype=int)
markers[peaks[:,0],peaks[:,1]] = np.arange(1, peaks.shape[0]+1)
# Visualize the watershed segments
labels = watershed(-density, markers, mask=density > 0.1 * density.max()) - 1
itp = RegularGridInterpolator((np.linspace(X_umap[:, 0].min(), X_umap[:, 0].max(),labels.shape[0]), np.linspace(X_umap[:, 1].min(), X_umap[:, 1].max(),labels.shape[1])),labels,method='nearest')
clusters = np.round(itp(X_umap)).astype(int)


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
np.random.seed(3)
# Create a 3D scatter plot
fig = plt.figure()
ax = fig.add_subplot(111)
c = np.random.rand(len(np.unique(clusters))  - np.min(clusters),3)
for i in np.unique(clusters):
    cluster_label = 'Noise' if i == -1 else f'Cluster {i}'
    idx_cluster_i = np.nonzero(clusters == i)[0]
    ax.scatter(chosen_umap[idx_cluster_i,0],chosen_umap[idx_cluster_i,1], c=c[i - np.min(clusters)], label = cluster_label)
    if i != -1:
        ax.text(np.mean(chosen_umap[idx_cluster_i,0]),np.mean(chosen_umap[idx_cluster_i,1]),i)
# ax.set_xlim((None,300))
# ax.legend()
fig.show()

In [ ]:
np.random.seed(3)
# Select 5 frames for each cluster
# idxs = np.array([np.random.choice(np.nonzero(clusters == i)[0],replace=False,size = 5) for i in np.unique(clusters)[-np.min(clusters):]])
idxs = chosen_idx[np.array([np.random.choice(np.nonzero(clusters == i)[0],replace=False,size = 5) for i in np.unique(clusters)])]

In [ ]:
# Get video idx and frame idx for each 5 frames of each cluster
idxs_videos = np.sum(idxs > n_frames_cumsum[:,np.newaxis,np.newaxis],axis=0)
idxs_frames = idxs - np.concatenate((np.array([0]),n_frames_cumsum))[idxs_videos]

In [ ]:
frames = np.empty((6,*idxs.shape, 1080, 1920, 3))
for v in np.unique(idxs_videos):
    frames_for_v = frames_included[v][idxs_frames[idxs_videos == v]]
    idxs_in_frames_array = np.nonzero(idxs_videos == v)
    for i in range(1,7):
        if os.path.isfile(os.path.join(videos[v],f'Output/videos_labeled_2d/output{i}.mp4')):
            vidcap = cv2.VideoCapture(os.path.join(videos[v],f'Output/videos_labeled_2d/output{i}.mp4'))
            for f_idx,f in enumerate(frames_for_v):
                vidcap.set(cv2.CAP_PROP_POS_FRAMES, f)
                success, frame = vidcap.read()
                if not success:
                    print(f"Couldn't read frame {f} in video {os.path.join(videos[v],f'Output/videos_labeled_2d/output{i}.mp4')}")
                frame[..., 0:3] = frame[..., ::-1]
                frames[i - 1, idxs_in_frames_array[0][f_idx], idxs_in_frames_array[1][f_idx]] = frame
        else:
            frames[i - 1, idxs_in_frames_array[0][f_idx], idxs_in_frames_array[1][f_idx]] = np.zeros((1080,1920,3))




In [ ]:
%matplotlib inline
plt.clf()
# for i in [3,2,1,6,7,4]:
for i in np.unique(clusters):
# for i in range(np.unique(clusters).size + np.min(clusters)):
    for j in range(5):
        fig, ax = plt.subplots(figsize=(16, 4))
        ax.axis('off')
        norm = mcolors.Normalize(vmin=-20, vmax=20)
        data = rotated_points_3d_velocity[idxs[i,j],:,:]
        data = np.concatenate((data,np.mean(data,axis=0,keepdims=True)),axis=0)
        data = np.concatenate((data,np.sqrt(np.einsum("ij,ij->i", data, data))[:,np.newaxis]),axis=1).transpose((1,0))
        cmap = plt.cm.RdYlBu
        table = ax.table(
            cellText=np.round(data,decimals=2),
            cellColours= cmap(norm(data)),
            rowLabels=['Velocity X','Velocity Y','Velocity Z','Velocity magnitude'],
            colLabels=config.animal['bodyparts'] + ['Mean'],
            loc='center',
            cellLoc='center',
            fontsize = 20
        )
        plt.show()



        fig, ax = plt.subplots(3,2,figsize = (30,15))
        for k in range(6):
            frame = frames[k,i,j]/255
            ax[int(k/2)][k%2].imshow(frame)
            ax[int(k/2)][k%2].set_title(f'Cluster {i}, {videos[idxs_videos[i,j]][30:]}, frame {frames_included[idxs_videos[i,j]][idxs_frames[i,j]]}, camera {k}')
            ax[int(k/2)][k%2].axis('off')
        fig.show()

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

xlim, ylim, zlim, _ = config.visualization['room_dimensions']

for i in range(n_clusters_opt):
    for j in range(5):
        # Create a 3D scatter plot
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        for bodyparts in config.visualization['skeleton'][::-1]:
            idx_bodyparts = []
            for bodypart in bodyparts:
                idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
            ax.plot(points_3d[0,idxs[i,j],idx_bodyparts,0],points_3d[0,idxs[i,j],idx_bodyparts,1],points_3d[0,idxs[i,j],idx_bodyparts,2], marker = 'o', ms=3)
        ax.set_title(f'Cluster {i}, frame {idxs[i,j]}')
        ax.set_xlim((0,xlim))
        ax.set_ylim((0,ylim))
        ax.set_zlim((0,zlim))

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

xlim, ylim, zlim, _ = config.visualization['room_dimensions']

for i in range(n_clusters_opt):
    for j in range(5):
        # Create a 3D scatter plot
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        for bodyparts in config.visualization['skeleton'][::-1]:
            idx_bodyparts = []
            for bodypart in bodyparts:
                idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
            ax.plot(rotated_points_3d[idxs[i,j],idx_bodyparts,0],rotated_points_3d[idxs[i,j],idx_bodyparts,1],rotated_points_3d[idxs[i,j],idx_bodyparts,2], marker = 'o', ms=3)
        ax.set_title(f'Cluster {i}, frame {idxs[i,j]}')
        # ax.set_xlim((0,xlim))
        # ax.set_ylim((0,ylim))
        # ax.set_zlim((0,zlim))

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

xlim, ylim, zlim, _ = config.visualization['room_dimensions']

points_3d_ = points_3d - points_3d[:,:,[idx_spinemid],:]
vector_body = points_3d_[:,:,[idx_neck],:] - points_3d_[:,:,[idx_tailbase],:]
normalized_velocity = rotated_points_3d_velocity/np.sqrt(np.einsum("ijkl,ijkl->j",vector_body,vector_body)[:,np.newaxis,np.newaxis])*100
normalized_velocity = np.concatenate((normalized_velocity,np.mean(normalized_velocity,axis=0,keepdims=True)),axis=0)
normalized_rotated_points_3d = rotated_points_3d[:-1]/np.sqrt(np.einsum("ijkl,ijkl->j",vector_body,vector_body)[:,np.newaxis,np.newaxis])[:-1]*100

for i in range(np.unique(clusters)):
    idx_cluster_i = np.nonzero(clusters == i)[0]
    mean_velocity = np.mean(normalized_velocity[idx_cluster_i,:,:], axis=0).transpose((1,0))
    var_velocity = np.var(normalized_velocity[idx_cluster_i,:,:], axis=0).transpose((1,0))

    fig, ax = plt.subplots(figsize=(16, 4))
    ax.axis('off')
    norm = mcolors.Normalize(vmin=0, vmax=5)
    cmap = plt.cm.Reds
    table = ax.table(
        cellText=np.round(mean_velocity,decimals=2),
        cellColours= cmap(norm(var_velocity)),
        rowLabels=['Velocity X','Velocity Y','Velocity Z','Velocity magnitude'],
        colLabels=config.animal['bodyparts'] + ['Mean'],
        loc='center',
        cellLoc='center',
        fontsize = 20
    )

    mean_points_3d = np.mean(normalized_rotated_points_3d[idx_cluster_i,:,:], axis=0)

    # Create a 3D scatter plot
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    for bodyparts in config.visualization['skeleton'][::-1]:
        idx_bodyparts = []
        for bodypart in bodyparts:
            idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
        ax.plot(mean_points_3d[idx_bodyparts,0],mean_points_3d[idx_bodyparts,1],mean_points_3d[idx_bodyparts,2], marker = 'o', ms=3)
    ax.set_title(f'Cluster {i}, Average')
    # ax.set_xlim((0,xlim))
    # ax.set_ylim((0,ylim))
    # ax.set_zlim((0,zlim))
        